# 03 — Feature Engineering

**Objective:** Prepare data for model training.

- Feature scaling (StandardScaler)
- SMOTE oversampling
- Train / Validation / Test split
- Save processed data to `data/processed/`

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import joblib

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.config import get_paths, get_model_config, get_train_config
from src.data.preprocess import (
    load_and_clean, drop_columns, encode_age_gender,
    label_encode_categoricals, log_transform_amount,
    temporal_split, scale_features
)
from src.data.augment import apply_smote

paths = get_paths()
cfg = get_model_config()['preprocessing']
target_col = cfg['target_col']
step_col = cfg['step_col']

## 1. Run Preprocessing Pipeline

In [ ]:
# Load and clean
df = load_and_clean()
df = drop_columns(df)
df = encode_age_gender(df)
df, encoders = label_encode_categoricals(df)
df = log_transform_amount(df)

print(f'Preprocessed shape: {df.shape}')
df.head()

## 2. Temporal Split

In [ ]:
df_train, df_test = temporal_split(df)

feature_cols = [c for c in df_train.columns if c not in [target_col, step_col]]
X_train = df_train[feature_cols].values
X_test = df_test[feature_cols].values
y_train = df_train[target_col].values
y_test = df_test[target_col].values

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

## 3. Feature Scaling (StandardScaler)

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

print(f'X_train_scaled: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}')
print(f'X_test_scaled:  mean={X_test_scaled.mean():.4f}, std={X_test_scaled.std():.4f}')

## 4. SMOTE Oversampling

In [ ]:
X_train_smote, y_train_smote = apply_smote(X_train_scaled, y_train)

print(f'After SMOTE: X={X_train_smote.shape}, y={y_train_smote.shape}')
print(f'Class 0: {(y_train_smote == 0).sum():,}')
print(f'Class 1: {(y_train_smote == 1).sum():,}')

## 5. Save Processed Data

In [ ]:
processed_dir = paths['data']['processed_dir']
os.makedirs(processed_dir, exist_ok=True)

# Save scaled data (without SMOTE — for AE training on normal-only)
np.save(os.path.join(processed_dir, 'X_train_scaled.npy'), X_train_scaled)
np.save(os.path.join(processed_dir, 'X_test_scaled.npy'), X_test_scaled)
np.save(os.path.join(processed_dir, 'y_train.npy'), y_train)
np.save(os.path.join(processed_dir, 'y_test.npy'), y_test)

# Save SMOTE-resampled data
np.save(os.path.join(processed_dir, 'X_train_smote.npy'), X_train_smote)
np.save(os.path.join(processed_dir, 'y_train_smote.npy'), y_train_smote)

# Save scaler and encoders
joblib.dump(scaler, os.path.join(processed_dir, 'scaler.pkl'))
joblib.dump(encoders, os.path.join(processed_dir, 'encoders.pkl'))
joblib.dump(feature_cols, os.path.join(processed_dir, 'feature_cols.pkl'))

print(f'All processed data saved to {processed_dir}')
for f in os.listdir(processed_dir):
    size = os.path.getsize(os.path.join(processed_dir, f))
    print(f'  {f:<30s} {size / 1e6:.2f} MB')